# Limpeza de Dados: Goodreads Dataset
Esta etapa trata exclusivamente da **higienização estrutural** do dataset bruto do Goodreads, preparando-o para consumo seguro nas etapas de Engenharia de Features e Análise. Nenhuma inteligência analítica é criada aqui — o foco é garantir que cada linha que avança no pipeline represente um registro real, confiável e bem tipado.

- **Escopo:** Remoção de ruídos, tratamento de nulos e correção de tipos primitivos (`date`, `float`)
- **Produto desta etapa:** `goodreads_books_cleaned.csv` — a fonte de verdade para os notebooks subsequentes

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Módulos customizados da pasta src/
from src import load_data
from src import save_dataset

---

## Carregando Dados

In [2]:
caminho = '../data/raw/Goodreads_books_with_genres.csv'
df_books = load_data(caminho, tipo_arquivo='csv')

Dados CSV carregados! Formato: (11127, 13)


---
### Critério de Descarte: Irrelevância para o Escopo Analítico

A coluna `isbn13` é um identificador de catálogo editorial sem valor analítico para os objetivos deste projeto — que visam padrões de recepção, popularidade e gênero literário. Sua presença apenas aumentaria a cardinalidade de identificadores únicos sem contribuir com nenhuma dimensão de análise.

In [3]:
# Dropando a coluna 'isbn13' que não é necessária para a análise
df_clean = df_books.drop(columns=['isbn13'])

---
## Padronização de colunas 
As colunas `Title` e `Book_id` são de estrema importância para este dataset, entretanto seus nomes se destacam das demais do schema, gerando anomalias visuais, uniformizar as colunas para `title` e `book_id`respectivamenete, poupará tempo de codagem, e deixará as visualisaçãoes e análises mais simples e claras. 

In [4]:
# Renomeando Coluna 'Title'
df_clean.rename(columns={'Title':'title'}, inplace=True)

In [5]:
# Renomeando Coluna 'Book Id'
df_clean.rename(columns={'Book Id':'book_id'}, inplace=True)

---
## Aplicando Limpeza de Missing Data 

### Filtro de Engajamento Real: Eliminação de Registros Fantasma

Livros com `average_rating = 0` ou `ratings_count = 0` não representam obras sem leitores — representam **registros incompletos** que nunca receberam interação verificável na plataforma. Mantê-los distorceria métricas de distribuição de notas e correlações de popularidade.

- **Regra aplicada:** Somente registros com engajamento comprovado (ao menos 1 avaliação) avançam no pipeline
- **Impacto:** Elimina o viés de registros cadastrados mas nunca consumidos pelo público

In [6]:
# Checando as colunas do DataFrame para confirmar a remoção da coluna 'isbn13'
df_clean.columns

Index(['book_id', 'title', 'Author', 'average_rating', 'isbn', 'language_code',
       'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date',
       'publisher', 'genres'],
      dtype='str')

In [7]:
# Remover linhas com valores nulos na coluna de gêneros
df_clean = df_clean.dropna(subset=['genres'])

# Aplicação do filtro de engajamento real utilizando a sintaxe otimizada .query()
df_clean = df_clean.query("average_rating > 0.0 and ratings_count > 0")

print(f"Limpeza concluída! Tamanho do dataset original: {len(df_books)}")
print(f"Tamanho do dataset limpo: {len(df_clean)}")
print(f"Total de linhas removidas: {len(df_books) - len(df_clean)}")


Limpeza concluída! Tamanho do dataset original: 11127
Tamanho do dataset limpo: 10965
Total de linhas removidas: 162


In [8]:
# Converte a coluna de texto para o tipo Data (datetime)
df_clean['publication_date'] = pd.to_datetime(df_clean['publication_date'], errors='coerce')

### Correção de Tipo: Integridade Temporal do Dataset

A coluna `publication_date` precisa ser tratada como um tipo temporal nativo para viabilizar qualquer análise de séries históricas, como evolução de popularidade por década ou sazonalidade de publicações. Armazenada como string, ela seria invisível para qualquer operação de filtragem ou agrupamento por período.

In [9]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 10965 entries, 0 to 11126
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   book_id             10965 non-null  int64         
 1   title               10965 non-null  str           
 2   Author              10965 non-null  str           
 3   average_rating      10965 non-null  float64       
 4   isbn                10965 non-null  str           
 5   language_code       10965 non-null  str           
 6   num_pages           10965 non-null  int64         
 7   ratings_count       10965 non-null  int64         
 8   text_reviews_count  10965 non-null  int64         
 9   publication_date    10963 non-null  datetime64[us]
 10  publisher           10965 non-null  str           
 11  genres              10965 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(6)
memory usage: 3.3 MB


---

## Salvando Novo Dataset Totatalmente Limpo

In [10]:
# Salvando o dataset limpo para a próxima etapa de análise
save_dataset(df=df_clean,nome_arquivo='goodreads_books_cleaned',tipo_arquivo='csv')

Sucesso! Ficheiro guardado em '..\data\processed\goodreads_books_cleaned.csv'


---
## Conclusão da Limpeza de Dados

O dataset bruto do Goodreads foi higienizado e está pronto para consumo analítico. As intervenções realizadas nesta etapa garantiram três propriedades fundamentais de qualidade:

- **Relevância:** Colunas sem valor para o escopo do projeto foram descartadas, reduzindo dimensionalidade desnecessária
- **Veracidade:** Registros sem engajamento verificável foram removidos para assegurar que métricas de recepção reflitam opiniões reais de leitores
- **Tipagem correta:** Campos temporais e numéricos foram convertidos para seus tipos nativos, habilitando operações analíticas precisas nas etapas seguintes